# SPARK Academy 2026 | Introduction to Machine Learning
## Notebook 3: Full ML Implementation Pipeline

**Instructor:** Aondona Moses
**Dataset:** Pima Indians Diabetes Dataset

---

### What you will learn:
1. The complete ML pipeline from raw data to final model
2. Implementing and comparing **multiple classifiers**
3. **Cross-validation** for robust evaluation
4. **Hyperparameter tuning** with GridSearchCV
5. **Feature importance** analysis
6. Making predictions on new patients
7. Saving and loading models

This notebook brings everything together | a production-quality ML workflow.

---

## 1. Setup & Data Preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_curve, auc, roc_auc_score)
import joblib
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print("All libraries loaded!")

In [ ]:
# Load and clean
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
           "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"]
df = pd.read_csv(url, names=columns)

# Clean missing values (zeros in clinical measurements)
zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
for col in zero_cols:
    df[col] = df[col].replace(0, np.nan).fillna(df[col].median())

# Split
X = df.drop("Outcome", axis=1)
y = df["Outcome"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training: {X_train.shape[0]} patients")
print(f"Test: {X_test.shape[0]} patients")
print(f"Features: {X.columns.tolist()}")

## 2. Model Comparison | 6 Classifiers

We will train and compare these algorithms:

| Model | Type | Strengths |
|-------|------|----------|
| Logistic Regression | Linear | Simple, interpretable, fast |
| K-Nearest Neighbors | Instance-based | No assumptions about data shape |
| Decision Tree | Tree-based | Easy to interpret, handles non-linear |
| Random Forest | Ensemble | Robust, handles overfitting |
| Gradient Boosting | Ensemble | Often best accuracy |
| Support Vector Machine | Kernel-based | Good for high-dimensional data |

In [ ]:
# Define models
models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "KNN (K=5)": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=5),
    "Random Forest": RandomForestClassifier(random_state=42, n_estimators=100),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42, n_estimators=100),
    "SVM": SVC(random_state=42, probability=True),
}

# Train and evaluate each model
results = []
for name, model in models.items():
    # Use scaled data for distance-based models, raw for tree-based
    if name in ["Decision Tree", "Random Forest", "Gradient Boosting"]:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc_score = roc_auc_score(y_test, y_prob)
    results.append({"Model": name, "Accuracy": acc, "AUC": auc_score})
    print(f"{name:25s}  Accuracy: {acc:.4f}  AUC: {auc_score:.4f}")

results_df = pd.DataFrame(results).sort_values("AUC", ascending=False)
print("\n=== Ranked by AUC ===")
print(results_df.to_string(index=False))

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
results_sorted = results_df.sort_values("Accuracy")
axes[0].barh(results_sorted["Model"], results_sorted["Accuracy"], color="#3498DB", edgecolor="white")
axes[0].set_xlabel("Accuracy", fontsize=12)
axes[0].set_title("Model Comparison: Accuracy", fontsize=14, fontweight="bold")
axes[0].set_xlim([0.6, 0.85])

# AUC
results_sorted_auc = results_df.sort_values("AUC")
axes[1].barh(results_sorted_auc["Model"], results_sorted_auc["AUC"], color="#E74C3C", edgecolor="white")
axes[1].set_xlabel("AUC", fontsize=12)
axes[1].set_title("Model Comparison: AUC", fontsize=14, fontweight="bold")
axes[1].set_xlim([0.6, 0.9])

plt.tight_layout()
plt.show()

## 3. Cross-Validation | More Robust Evaluation

In [ ]:
# 5-fold stratified cross-validation
print("=== 5-Fold Cross-Validation ===")
print(f"{'Model':25s}  {'Mean Acc':>8}  {'Std':>6}")
print("-" * 45)

cv_results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    data = X_train if name in ["Decision Tree", "Random Forest", "Gradient Boosting"] else pd.DataFrame(X_train_scaled, columns=X.columns)
    scores = cross_val_score(model, data, y_train, cv=cv, scoring="accuracy")
    cv_results.append({"Model": name, "Mean": scores.mean(), "Std": scores.std(), "Scores": scores})
    print(f"{name:25s}  {scores.mean():>8.4f}  {scores.std():>6.4f}")

print("\nCross-validation gives a more reliable estimate than a single train/test split")

In [ ]:
# Visualize cross-validation results
cv_df = pd.DataFrame(cv_results).sort_values("Mean", ascending=True)

plt.figure(figsize=(10, 5))
for i, row in cv_df.iterrows():
    plt.barh(row["Model"], row["Mean"], xerr=row["Std"], color="#2ECC71",
             capsize=5, edgecolor="white", alpha=0.8)
plt.xlabel("Accuracy (mean ± std)", fontsize=12)
plt.title("5-Fold Cross-Validation Results", fontsize=14, fontweight="bold")
plt.xlim([0.6, 0.85])
plt.tight_layout()
plt.show()

## 4. Hyperparameter Tuning | GridSearchCV

In [ ]:
# Tune the best model (Random Forest)
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 5, 7, 10],
    "min_samples_split": [2, 5, 10],
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=0
)

grid_search.fit(X_train, y_train)

print("=== Grid Search Results ===")
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV AUC: {grid_search.best_score_:.4f}")

# Evaluate on test set
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
y_prob_best = best_model.predict_proba(X_test)[:, 1]

print(f"\nTest Accuracy: {accuracy_score(y_test, y_pred_best):.4f}")
print(f"Test AUC: {roc_auc_score(y_test, y_prob_best):.4f}")

In [ ]:
# Final classification report
print("=== Final Model: Tuned Random Forest ===")
print(classification_report(y_test, y_pred_best, target_names=["No Diabetes", "Diabetes"]))

## 5. Feature Importance Analysis

In [ ]:
# Feature importance from Random Forest
importances = best_model.feature_importances_
feat_imp = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importances
}).sort_values("Importance", ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(feat_imp["Feature"], feat_imp["Importance"], color="#E88A1A", edgecolor="white")
plt.xlabel("Importance", fontsize=12)
plt.title("Random Forest: Feature Importance", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("Top 3 most important features:")
for _, row in feat_imp.tail(3).iterrows():
    print(f"  {row['Feature']}: {row['Importance']:.4f}")

## 6. ROC Curves | All Models

In [ ]:
# Plot ROC curves for all models
plt.figure(figsize=(8, 7))

for name, model in models.items():
    if name in ["Decision Tree", "Random Forest", "Gradient Boosting"]:
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = model.predict_proba(X_test_scaled)[:, 1]

    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC={roc_auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random (AUC=0.5)")
plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate", fontsize=12)
plt.title("ROC Curves — All Models", fontsize=14, fontweight="bold")
plt.legend(loc="lower right", fontsize=10)
plt.tight_layout()
plt.show()

## 7. Predict on New Patients

In [ ]:
# Predict for new patients
new_patients = pd.DataFrame({
    "Pregnancies": [2, 0, 5],
    "Glucose": [140, 95, 180],
    "BloodPressure": [85, 70, 90],
    "SkinThickness": [30, 20, 35],
    "Insulin": [150, 80, 200],
    "BMI": [32.0, 22.5, 38.0],
    "DiabetesPedigreeFunction": [0.5, 0.2, 0.8],
    "Age": [45, 25, 55]
})

predictions = best_model.predict(new_patients)
probabilities = best_model.predict_proba(new_patients)[:, 1]

print("=== New Patient Predictions ===")
for i, (_, row) in enumerate(new_patients.iterrows()):
    status = "DIABETIC" if predictions[i] == 1 else "HEALTHY"
    print(f"Patient {i+1}: Age={row['Age']:.0f}, Glucose={row['Glucose']:.0f}, BMI={row['BMI']:.1f}")
    print(f"  Prediction: {status} (probability: {probabilities[i]:.1%})")
    print()

## 8. Save & Load Model

In [ ]:
# Save the model and scaler
joblib.dump(best_model, "diabetes_model.pkl")
joblib.dump(scaler, "diabetes_scaler.pkl")
print("Model saved to diabetes_model.pkl")
print("Scaler saved to diabetes_scaler.pkl")

# Load and verify
loaded_model = joblib.load("diabetes_model.pkl")
loaded_scaler = joblib.load("diabetes_scaler.pkl")
verify_pred = loaded_model.predict(new_patients)
print(f"\nVerification: loaded model predictions match: {np.array_equal(verify_pred, predictions)}")

## 9. Summary | The Complete ML Pipeline

```
1. Load Data           →  pd.read_csv()
2. Explore             →  df.describe(), visualizations
3. Clean               →  Handle missing values, outliers
4. Feature Engineering →  Scale, encode, create new features
5. Split               →  train_test_split (80/20)
6. Train Models        →  Multiple algorithms
7. Evaluate            →  Cross-validation, confusion matrix, ROC
8. Tune                →  GridSearchCV for best hyperparameters
9. Interpret           →  Feature importance, coefficients
10. Deploy             →  Save model, predict on new data
```

### Key Results on Diabetes Dataset:
- **Best model:** Tuned Random Forest
- **Key features:** Glucose, BMI, Age, DiabetesPedigreeFunction
- **Clinical insight:** Glucose level is the single strongest predictor of diabetes

---
*SPARK Academy 2026 — Train for Change, From Science to Practice* 🚀